# 03 · Apuração parcial na ordem real do TSE

Parte 18. Pergunta: o modelo `bb_slx` prevê o resultado final melhor que a contagem bruta e que o swing uniforme? O modelo é ajustado nos municípios já apurados, com x % dos votos válidos já recebidos. Os votos seguem a **ordem real em que o TSE recebeu os boletins de urna (BU)**. Regras e resultado em [doc 11](../docs/pressupostos/11-apuracao-parcial.md).

Fonte: `data/interim/tse_bweb.parquet` (seção a seção, 2022 e 2026; coletor `src/coleta/tse_bweb.py`) e `data/interim/tse_bweb_1t.parquet` (1º turno do mesmo ano, por seção). Funções em `src/apuracao.py`. Tabelas em `results/tables/apuracao_*.csv`, figuras em `results/figures/apuracao_*.png`.

**Regra.** Nasceu com duas linhas de base (bruta e swing uniforme). O swing do 1º turno do mesmo ano (`swing_1t`) entrou depois, como terceira linha de base formal, porque o modelo usa o 1º turno do mesmo ano: bater só quem não o usa é uma comparação frouxa. A regra ficou mais exigente, e a mudança foi feita depois de ver o resultado (registrada no [doc 11](../docs/pressupostos/11-apuracao-parcial.md)).
- Cortes de 5, 10, 25, 50, 67 e 90 % dos votos válidos recebidos no Brasil (doméstico), em 2022 e em 2026.
- O modelo é **útil** se o erro absoluto do Brasil **e** o MAE regional (5 regiões) forem menores que os de **todas** as linhas de base: (i) contagem bruta, (ii) swing uniforme e (iii) swing uniforme do 1º turno do mesmo ano.
- As duas condições devem valer em **pelo menos 4 dos 6 cortes, em cada ano**.
- Alvo: o resultado final real.

**Desenho.**
- Por município, (y_c, n_c) = votos de Lula e válidos das seções recebidas até o corte. n_u = aptos das seções ainda não recebidas × (válidos ÷ aptos do 1º turno do mesmo ano no município).
- Modelo: a família `bb_slx` (beta-binomial com SLX), mas **o desenho é refeito em cada (ano, corte) só com os municípios já apurados** (`src/desenho_parcial.py`, regras R1 a R7 do [doc 11](../docs/pressupostos/11-apuracao-parcial.md), seção 9): covariáveis (inclusive as do 1º turno do mesmo ano) e fusão de regiões (R1), corte de Conley (R2), dispersão (R4), SLX (R5) e σ_UF (R7). O desenho final do `02-MLG` (escolhido com o resultado de 2026) **não é lido**: usá-lo vazava o resultado final para as estimativas parciais. O ajuste usa (y_c, n_c) dos municípios com n_c > 0 e o modelo prevê π̂ do restante. O desenho de cada corte fica em `results/tables/apuracao_desenho.csv`.
- 2026 usa `MAPA_2026_PRE`. 2022 usa `MAPA_FIT_2022` (covariáveis anteriores a 2022, sem `delta_comp`).
- Covariáveis do 1º turno do mesmo ano: `lag1t` (logit de Lula ÷ (Lula + Bolsonaro), com Flávio no lugar de Jair em 2026) e `outros_1t` (parcela dos outros candidatos). São conhecidas antes do 2º turno (regra de informação, doc 09). Na apuração, medem-se por seção, separadamente nas seções já recebidas e no restante.
- A padronização usa só as linhas de treino. Total = Σ y_c + Σ π̂·n_u, por região e Brasil.
- (i) contagem bruta: share já apurado.
- (ii) swing uniforme: swing nacional médio dos apurados contra o 2º turno anterior (2018 para 2022; 2022 para 2026). Soma-se ao share anterior de cada município não apurado.
- (iii) swing uniforme do 1º turno (`swing_1t`): swing do 1º para o 2º turno das partes apuradas, aplicado ao 1º turno das partes restantes.
- Boa Esperança do Norte (MT, código TSE 73709) não está no parquet do modelo. Seus votos apurados entram nos quatro métodos e no alvo. O restante usa o share do método na UF.
- Intervalos de 95 %: `simular_votos_grupo` (β ~ N(β̂, Σ Conley), choque de UF σ̂·T_k, beta-binomial; B = 1000) sobre o restante, somado ao já apurado. O corte de Conley (75 a 150 km) e σ_UF (0,011 a 0,027) vêm do desenho de cada corte, não da Parte 15 nem da Parte 16. O choque de UF é σ̂·T_k, com T de Student e k graus de liberdade, k = número de UFs que estimam σ_UF no desenho do corte (24 a 26): σ̂ é a RMS de k resíduos de UF e tratá-lo como conhecido estreitaria o intervalo.
- ESF, krigagem e CAR não são retestados por corte: o `02-MLG` os rejeitou e eles só acrescentariam termos (R6).

In [1]:
import sys, json, warnings
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
# Silencia avisos de bibliotecas, mas mantém o aviso de covariância não PSD: ele indica problema no ajuste.
warnings.filterwarnings("ignore")
warnings.filterwarnings("always", message="Covariância não PSD")
from statsmodels.tools.sm_exceptions import PerfectSeparationError
# Roda da raiz do repositório ou de notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import desenho_parcial as DP, especificacoes as E, espacial as S, backtest as BT, apuracao as A, graficos as G, pressupostos as PR, dados as D, saidas as SA, apuracao_diagnostico as AD
from src.especificacoes import Candidato, Termos
TAB, FIG = ROOT / "results/tables", ROOT / "results/figures"
for p in (TAB, FIG): p.mkdir(parents=True, exist_ok=True)
SEMENTE, B = 7, 1000   # o corte de Conley e σ_UF saem do desenho de cada corte (src/desenho_parcial.py)
ANOS = (2022, 2026)
BD = D.carregar_base(ROOT)   # municípios, geometria na mesma ordem e pesos Queen (compartilhado com o notebook 02-MLG)
dfm, W = BD.df, BD.W
G.configurar(dfm, FIG)
lat, lon = dfm["lat"].to_numpy(float), dfm["lon"].to_numpy(float)
cd_modelo = dfm["id_municipio_tse"].astype("int64")
# O desenho (termos, dispersão, SLX, corte de Conley e σ_UF) é refeito em cada (ano, corte) só com os municípios já apurados
# (src/desenho_parcial.py, regras R1 a R7). Nada vem de final_decisao.json nem de final_cobertura.csv do 02-MLG.
LIGACAO = "logit"
MAPAS = {2026: E.MAPA_2026, 2022: BT.MAPA_FIT_2022}
# delta_comp de 2026 é a variação 2018→2022 do comparecimento no 2º turno; o análogo de 2022 exigiria 2014,
# que não está na base. Em 2022 ele não é candidata (DP.candidatas_do_ano), nem como W·x.
# Seção a seção, com o instante de recebimento do BU: é a base da ordem real de chegada.
secoes = pd.read_parquet(ROOT / "data/interim/tse_bweb.parquet")
# Covariáveis lag1t/outros_1t do 1º turno medidas por parte (apurada e restante), não só no município; False reproduz a versão municipal.
PARTES_1T = True
S1T = pd.read_parquet(ROOT / "data/interim/tse_bweb_1t.parquet") if PARTES_1T else None


## Taxa de válidos do 1º turno como estimador de n_u

`n_u` precisa de uma taxa válidos ÷ aptos anterior ao 2º turno. A leitura literal (só Lula + adversário no 1º turno) ignora o voto de quem escolheu um terceiro candidato no 1º turno e vota nos dois no 2º; a alternativa é o total de válidos do 1º turno. A tabela compara as duas contra o n real do 2º turno (informação só para escolher o estimador, não entra nos métodos).

In [2]:
# n_u = aptos das seções não recebidas × (válidos ÷ aptos do 1º turno). A taxa é do 1º turno por ser definida antes do 2º turno.
# O voto dos outros candidatos migra no 2º turno: a taxa só com Lula e o adversário ignora quem votou num terceiro candidato.
# Leitura: soma_estimada_sobre_real perto de 1 e erro municipal baixo indicam a melhor taxa.
L = []
for ano in ANOS:
    aptos1 = dfm[f"aptos_1t_{ano}"].astype(float)
    # n real do 2º turno sobre aptos do 1º. Serve só para comparar as taxas. Não entra nos métodos.
    real_r = dfm[f"n_2t_{ano}"] / aptos1
    for nome, num in (("dois_candidatos_1t", dfm[f"votos_pt_1t_{ano}"] + dfm[f"votos_bolsonaro_1t_{ano}"]),
                      ("todos_validos_1t", dfm[f"validos_1t_{ano}"])):
        r = num.astype(float) / aptos1
        L.append({"ano": ano, "taxa": nome, "soma_estimada_sobre_real": float((r * aptos1).sum() / (real_r * aptos1).sum()),
                  "mape_municipal_pct": float((abs(r - real_r) / real_r).mean() * 100)})
TAXAS = pd.DataFrame(L); TAXAS.to_csv(TAB / "apuracao_taxa_nu.csv", index=False)
display(SA.tabela(TAXAS.assign(ano=TAXAS["ano"].astype(str)), "Taxa de válidos do 1º turno como estimador de n_u, contra o n real do 2º turno", numero="1", casas=3,
                  indice=False, rotulos={"ano": "Ano", "taxa": "Taxa", "soma_estimada_sobre_real": "Soma estimada / real", "mape_municipal_pct": "Erro médio por município (%)"},
                  casas_por_coluna={"mape_municipal_pct": 1}))

Ano,Taxa,Soma estimada / real,Erro médio por município (%)
2022,dois_candidatos_1t,"0,914","6,6"
2022,todos_validos_1t,"0,997","2,1"
2026,dois_candidatos_1t,"0,887","9,5"
2026,todos_validos_1t,"0,962","4,1"


## Ordem real e cortes

Os cortes seguem a curva acumulada por instante de recebimento do BU (horário de Brasília). A virada de 2022 (Lula passa de Bolsonaro) está validada no coletor em 18:44 a 68,1 % recebido. O % efetivo de cada corte é um pouco maior que o nominal, porque seções do mesmo instante entram juntas.

In [3]:
# Para cada corte: instante do BU em que a fatia acumulada é atingida e % efetivo de válidos recebidos.
L = []
for ano in ANOS:
    s = secoes[secoes.ano == ano]
    for pct in A.CORTES:
        rec = A.recebidas_ate(s, pct)
        L.append({"ano": ano, "corte": pct, "instante": A.instante_do_corte(s, pct).strftime("%d/%m %H:%M"), "pct_efetivo": A.pct_efetivo(s, rec)})
CORTES = pd.DataFrame(L)
tab_cortes = CORTES.pivot(index="corte", columns="ano", values=["instante", "pct_efetivo"])
tab_cortes.columns = [f"{'Instante do BU' if a == 'instante' else '% efetivo'} {b}" for a, b in tab_cortes.columns]
tab_cortes = tab_cortes[[f"{k} {a}" for a in ANOS for k in ("Instante do BU", "% efetivo")]]
tab_cortes = tab_cortes.apply(lambda c: c.astype(float) if c.name.startswith("%") else c)
display(SA.tabela(tab_cortes.rename_axis("Corte (%)"), "Cortes de apuração: instante do boletim (Brasília) e % efetivo dos válidos", numero="2", casas=1))

,Instante do BU 2022,% efetivo 2022,Instante do BU 2026,% efetivo 2026
Corte (%),,,,
5,30/10 17:25,"5,0",25/10 19:28,"5,0"
10,30/10 17:33,"10,0",25/10 19:38,"10,0"
25,30/10 17:55,"25,0",25/10 20:02,"25,0"
50,30/10 18:26,"50,0",25/10 20:26,"50,0"
67,30/10 18:42,"67,0",25/10 20:41,"67,0"
90,30/10 19:08,"90,0",25/10 21:01,"90,0"


## Ajuste, estimativas e intervalos por corte

Os intervalos saem duas vezes: com n_u fixo e com n_u simulado (regra do [doc 11](../docs/pressupostos/11-apuracao-parcial.md), seção 3). Os 6 cortes são aninhados no tempo e as 6 unidades (5 regiões e Brasil) são correlacionadas: a tabela de cortes cobertos por unidade e ano é descritiva, não um teste de calibração, e não se soma entre unidades.

Cada corte tem o desenho próprio (ver `apuracao_desenho.csv`): `lag1t` entra em todos, a dispersão é por região em 8 dos 12 cortes (única em 2022 e 2026 a 5% e a 10%) e o SLX é aceito em 7 (2022 de 25 a 90%; 2026 de 50 a 90%); um ajuste que falha numa dobra do LOSO é pulado e registrado, e um corte sem desenho conta como falha do modelo. Quando o θ beta-binomial de um grupo vai ao limite binomial (θ ≥ 1e5 em todas as linhas do grupo), ele não é identificado: fixa-se no teto (1e9, com teste da razão de verossimilhanças como guarda) e sai dos contrastes de Wald e da contagem do BIC. Foi o que fez convergir o ajuste de 2026 a 10% (|g| 1,4e-5); o Wald A dá p = 0,0708 e a decisão continua dispersão única.

In [4]:
LIN, LIN_REG, LIN_IC, LIN_ICN, LIN_SD, LIN_REP, LIN_NU, FALHAS = [], [], [], [], [], [], [], []
PI_CORTE = {}
# Desenho de cada (ano, corte) a partir só do apurado: bases, depois o desenho com cache em results/cv.
DADOS = {}
for ano in ANOS:
    s = secoes[secoes.ano == ano].reset_index(drop=True)
    for pct in A.CORTES:
        bm = A.montar_base(s, dfm, ano, A.recebidas_ate(s, pct), secoes_1t=S1T).reindex(cd_modelo)
        treino = A.sobrescritas_1t(bm, MAPAS[ano])[0] if PARTES_1T else None
        DADOS[(ano, pct)] = DP.dados_do_corte(dfm, MAPAS[ano], W, np.nan_to_num(bm.y_c.to_numpy(float)), np.nan_to_num(bm.n_c.to_numpy(float)), treino)
FALHAS_D = {}
DESENHOS = DP.desenhos_por_corte(DADOS, pasta=ROOT / "results/cv", falhas=FALHAS_D)
LIN_D = []
for (ano, pct), dados_c in DADOS.items():
    d = DESENHOS.get((ano, pct))
    if d is None:
        LIN_D.append({"ano": ano, "corte": pct, "erro": FALHAS_D.get((ano, pct), "")})
        continue
    t = d.termos
    LIN_D.append({"ano": ano, "corte": pct, "vars": ",".join(t.vars), "regiao_fundir": ";".join("+".join(g) for g in t.regiao_fundir),
                  "bloco_grupos": ";".join("+".join(g) for g in t.bloco_grupos), "disp": t.disp, "slx": ",".join(t.slx),
                  "disp_aceita": d.aceitos["disp"], "slx_aceito": d.aceitos["slx"], "corte_km": d.corte_km, "sigma_uf": d.sigma_uf,
                  "limite_pp": d.limite, "mae_r1_pp": d.mae_pp["r1"], "mae_disp_pp": d.mae_pp.get("disp"), "mae_slx_pp": d.mae_pp.get("slx"),
                  "mae_final_pp": d.mae_pp["final"], "n_municipios": d.n_municipios, "n_ufs_loso": d.n_ufs_loso,
                  "ufs_puladas": ";".join(f"{k}:{'+'.join(v)}" for k, v in d.extra.get("ufs_puladas", {}).items() if v),
                  "corte_r1_km": d.extra.get("corte_r1_km"), "piso_corte_aplicado": any(d.extra.get("piso_aplicado", {}).values()),
                  "tempo_s": d.tempo_s})
pd.DataFrame(LIN_D).to_csv(TAB / "apuracao_desenho.csv", index=False)
CAND_CORTE = {}   # (ano, corte) -> (candidato, df com W·x do desenho, mapa, corte de Conley); reaproveitado pela sensibilidade a n_u
for ano in ANOS:
    s = secoes[secoes.ano == ano].reset_index(drop=True)
    for pct in A.CORTES:
        rec = A.recebidas_ate(s, pct)
        base = A.montar_base(s, dfm, ano, rec, secoes_1t=S1T)
        real = A.alvo(base)
        info = {"ano": ano, "corte": pct, "pct_efetivo": A.pct_efetivo(s, rec)}
        assert PARTES_1T   # o swing do 1º turno usa share_c, que só vem com as partes do 1º turno
        ests = {"bruta": A.estimativa_bruta(base), "swing": A.estimativa_swing(base), "swing_1t": AD.estimativa_swing_1t(base)}
        LIN_NU.append({**info, "n_u_estimado": float(base.n_u.sum()), "restante_real": float((base.n_final - base.n_c).sum()),
                       "razao": float(base.n_u.sum() / (base.n_final - base.n_c).sum())})
        pi = None
        try:
            bm = base.reindex(cd_modelo)
            des = DESENHOS.get((ano, pct))
            if des is None:
                raise np.linalg.LinAlgError(f"desenho indisponível: {FALHAS_D.get((ano, pct), '')}")
            df2, mapa = S.adicionar_slx(dfm, MAPAS[ano], list(des.termos.slx), W)
            prois = {k: v for k, v in mapa.items() if k in des.termos.vars and f"2t_{ano}" in v}
            assert not prois, prois   # nenhuma covariável do modelo vem do 2º turno do próprio ano
            cand = Candidato("parcial", "betabin", LIGACAO, des.termos)
            CAND_CORTE[(ano, pct)] = (cand, df2, mapa, des.corte_km)
            treino, resto = A.sobrescritas_1t(bm, mapa) if PARTES_1T else (None, None)
            pm = A.ajustar_parcial(cand, df2, mapa, bm.y_c.to_numpy(), bm.n_c.to_numpy(), bm.n_u.to_numpy(), lat, lon, des.corte_km, treino, resto)
            pi = pd.Series(pm.pi, index=bm.index)
            ests["modelo"] = A.estimativa_modelo(base, pi)
            assert des.gl_sigma_uf is not None   # cache sem a contagem de UFs do σ: apagar e refazer o desenho
            ic = A.intervalos(pm, bm, base, A.share_uf_modelo(base, pi), LIGACAO, des.sigma_uf, B, SEMENTE, gl_uf=des.gl_sigma_uf)
            ic = ic.assign(real=real.reindex(ic.index), estimativa=ests["modelo"].reindex(ic.index))
            ic["dentro"] = (ic.real >= ic.q025) & (ic.real <= ic.q975)
            for r_, x in ic.iterrows():
                LIN_IC.append({**info, "regiao": r_, "estimativa_pct": 100 * x.estimativa, "q025_pct": 100 * x.q025, "q975_pct": 100 * x.q975,
                               "real_pct": 100 * x.real, "largura_pp": 100 * (x.q975 - x.q025), "dentro_ic95": bool(x.dentro)})
            fk = A.fatores_nu(base, B, SEMENTE)   # n_u simulado: mesma semente e B, só seções já recebidas
            icn = A.intervalos(pm, bm, base, A.share_uf_modelo(base, pi), LIGACAO, des.sigma_uf, B, SEMENTE, fatores=fk, gl_uf=des.gl_sigma_uf)
            icn = icn.assign(real=real.reindex(icn.index))
            for r_, x in icn.iterrows():
                LIN_ICN.append({**info, "regiao": r_, "q025_pct": 100 * x.q025, "q975_pct": 100 * x.q975, "real_pct": 100 * x.real,
                                "largura_pp": 100 * (x.q975 - x.q025), "dentro_ic95": bool(x.real >= x.q025 and x.real <= x.q975)})
            LIN_SD.append({**info, "sd_uf_nu": A.sd_uf_nu(base)})
        except np.linalg.LinAlgError as e:   # matriz singular no ajuste conta como derrota do modelo; qualquer outro erro é bug e interrompe
            FALHAS.append({**info, "erro": repr(e)[:200]})
        PI_CORTE[(ano, pct)] = pi
        for met, est in ests.items():
            eb, mr = A.erros(est, real)
            LIN.append({**info, "metodo": met, "estimativa_brasil_pct": 100 * est["Brasil"], "real_brasil_pct": 100 * real["Brasil"],
                        "erro_brasil_com_sinal_pp": 100 * (est["Brasil"] - real["Brasil"]), "erro_brasil_pp": eb, "mae_regional_pp": mr})
            for r_ in real.index.drop("Brasil"):
                LIN_REG.append({**info, "metodo": met, "regiao": r_, "estimativa_pct": 100 * est[r_], "real_pct": 100 * real[r_], "erro_pp": 100 * (est[r_] - real[r_])})
        rep = A.representatividade(base, pi)
        for _, x in rep.iterrows(): LIN_REP.append({**info, **x.to_dict()})
TAB_ERROS = pd.DataFrame(LIN)

In [5]:
METODOS = A.METODOS
IC = pd.DataFrame(LIN_IC)
ICN = pd.DataFrame(LIN_ICN); ICN.to_csv(TAB / "apuracao_ic_nu_simulado.csv", index=False)
pd.DataFrame(LIN_SD).to_csv(TAB / "apuracao_sd_uf_nu.csv", index=False)
# n_u fixo (IC antigo) contra n_u simulado: cortes cobertos de 6, por unidade (5 regiões + Brasil) e ano
CHAVE = ["ano", "corte", "regiao"]
J = IC[CHAVE + ["dentro_ic95", "largura_pp"]].merge(ICN[CHAVE + ["dentro_ic95", "largura_pp"]], on=CHAVE, suffixes=("_fixo", "_sim"))
J["delta_largura_pp"] = J.largura_pp_sim - J.largura_pp_fixo
COB_UNI = J.groupby(["regiao", "ano"]).agg(cortes=("corte", "size"), cobertos_n_u_fixo=("dentro_ic95_fixo", "sum"), cobertos_n_u_simulado=("dentro_ic95_sim", "sum"),
                                         mediana_delta_largura_pp=("delta_largura_pp", "median")).reset_index().rename(columns={"regiao": "unidade"})
COB_UNI.to_csv(TAB / "apuracao_cobertura_unidade.csv", index=False)
COBERTURA = IC.groupby(["ano", "corte"]).agg(dentro=("dentro_ic95", "sum"), total=("dentro_ic95", "size"), largura_brasil_pp=("largura_pp", "last")).reset_index()
COBERTURA["cobertura"] = COBERTURA.dentro / COBERTURA.total
cob = COBERTURA.set_index(["ano", "corte"])["cobertura"]
TAB_ERROS["cobertura_ic95"] = [cob.get((a, c), np.nan) if m == "modelo" else np.nan for a, c, m in zip(TAB_ERROS.ano, TAB_ERROS.corte, TAB_ERROS.metodo)]
for ano in ANOS:
    TAB_ERROS[TAB_ERROS.ano == ano].to_csv(TAB / f"apuracao_{ano}.csv", index=False)
pd.DataFrame(LIN_REG).to_csv(TAB / "apuracao_regioes.csv", index=False)
IC.to_csv(TAB / "apuracao_ic.csv", index=False)
REP = pd.DataFrame(LIN_REP); REP.to_csv(TAB / "apuracao_representatividade.csv", index=False)
NU = pd.DataFrame(LIN_NU); NU.to_csv(TAB / "apuracao_n_restante.csv", index=False)
if FALHAS:
    display(SA.tabela(pd.DataFrame(FALHAS), "Ajustes que falharam (contam como derrota do modelo)", numero="F", indice=False))
RT = {"bruta": "bruta", "swing": "swing", "swing_1t": "swing_1t", "modelo": "modelo"}
for k, ano in enumerate(ANOS):
    t = TAB_ERROS[TAB_ERROS.ano == ano]
    ex = t.pivot(index="corte", columns="metodo", values=["erro_brasil_pp", "mae_regional_pp"])[[(c, m) for c in ("erro_brasil_pp", "mae_regional_pp") for m in METODOS]]
    ex.columns = [("Erro Brasil " if c == "erro_brasil_pp" else "MAE regional ") + m for c, m in ex.columns]
    display(SA.tabela(ex.rename_axis("Corte (%)"), f"{ano}: erro absoluto do Brasil e MAE regional por corte e método (p.p.)", numero=f"{3 + k}", casas=2))
display(SA.tabela(COB_UNI.assign(ano=COB_UNI["ano"].astype(str)), "Cortes cobertos pelo IC 95% (de 6), por unidade e ano: n_u fixo contra n_u simulado", numero="5", casas=3,
                  indice=False, inteiros=["cortes", "cobertos_n_u_fixo", "cobertos_n_u_simulado"],
                  rotulos={"unidade": "Unidade", "ano": "Ano", "cortes": "Cortes", "cobertos_n_u_fixo": "Cobertos (n_u fixo)", "cobertos_n_u_simulado": "Cobertos (n_u simulado)",
                           "mediana_delta_largura_pp": "Mediana da variação de largura (p.p.)"}))


,Erro Brasil bruta,Erro Brasil swing,Erro Brasil swing_1t,Erro Brasil modelo,MAE regional bruta,MAE regional swing,MAE regional swing_1t,MAE regional modelo
Corte (%),,,,,,,,
5,"4,57","0,60","0,39","0,08","0,77","3,45","0,39","0,14"
10,"3,05","1,53","0,24","0,13","0,52","3,43","0,24","0,10"
25,"2,20","1,62","0,13","0,15","0,74","2,88","0,13","0,10"
50,"1,23","1,09","0,08","0,10","0,66","1,81","0,07","0,06"
67,"0,93","0,61","0,06","0,07","0,60","1,16","0,05","0,04"
90,"0,38","0,13","0,02","0,02","0,31","0,34","0,02","0,01"


,Erro Brasil bruta,Erro Brasil swing,Erro Brasil swing_1t,Erro Brasil modelo,MAE regional bruta,MAE regional swing,MAE regional swing_1t,MAE regional modelo
Corte (%),,,,,,,,
5,"0,97","1,19","0,27","0,16","1,78","1,49","0,52","0,16"
10,"1,48","0,53","0,23","0,11","1,04","1,00","0,50","0,13"
25,"2,37","0,39","0,20","0,08","0,83","0,46","0,42","0,09"
50,"1,67","0,53","0,14","0,04","0,60","0,33","0,27","0,04"
67,"0,95","0,40","0,11","0,02","0,49","0,22","0,17","0,02"
90,"0,22","0,14","0,04","0,00","0,15","0,07","0,04","0,01"


Unidade,Ano,Cortes,Cobertos (n_u fixo),Cobertos (n_u simulado),Mediana da variação de largura (p.p.)
Brasil,2022,6,6,6,"0,058"
Brasil,2026,6,6,6,"0,149"
Centro-Oeste,2022,6,6,6,"-0,007"
Centro-Oeste,2026,6,6,6,"0,010"
Nordeste,2022,6,6,6,"0,018"
Nordeste,2026,6,6,6,"0,020"
Norte,2022,6,6,6,"0,102"
Norte,2026,6,6,6,"0,144"
Sudeste,2022,6,6,6,"0,009"
Sudeste,2026,6,6,6,"0,046"


## A regra aplicada

In [6]:
# Cortes de 5, 10, 25, 50, 67 e 90 %: cada corte vale a partir do primeiro instante de chegada cuja fatia acumulada de válidos alcança x %.
# Seções do mesmo instante entram juntas, por isso o % efetivo fica um pouco acima do nominal.
REGRA = A.aplicar_regra(TAB_ERROS)
det = []
for (ano, c), p in TAB_ERROS.groupby(["ano", "corte"]):
    p = p.set_index("metodo")
    if "modelo" not in p.index:
        det.append({"ano": ano, "corte": c, "vence_brasil": False, "vence_regional": False, "vence": False}); continue
    # Vence o corte quem tem erro menor que a bruta, o swing E o swing do 1º turno, no Brasil (vb) e no MAE regional (vr). O modelo precisa vencer nos dois.
    vb = bool((p.loc["modelo", "erro_brasil_pp"] < p.drop(index="modelo")["erro_brasil_pp"]).all())
    vr = bool((p.loc["modelo", "mae_regional_pp"] < p.drop(index="modelo")["mae_regional_pp"]).all())
    det.append({"ano": ano, "corte": c, "vence_brasil": vb, "vence_regional": vr, "vence": vb and vr})
DET = pd.DataFrame(det)
# Útil = vitórias em pelo menos 4 dos 6 cortes, em cada ano. Regra fixada antes do resultado. Ajuste que falhou conta como derrota.
REGRA["veredito"] = np.where(REGRA.util, "modelo útil", "modelo não útil")
DET.to_csv(TAB / "apuracao_regra_cortes.csv", index=False); REGRA.to_csv(TAB / "apuracao_regra.csv", index=False)
det_x = DET.assign(**{c: DET[c].map({True: "sim", False: "não"}) for c in ("vence_brasil", "vence_regional", "vence")}).assign(ano=DET["ano"].astype(str), corte=DET["corte"].astype(str))
display(SA.tabela(det_x, "O modelo vence a contagem bruta, o swing e o swing do 1º turno? (menor erro do Brasil e menor MAE regional)", numero="6", indice=False,
                  rotulos={"ano": "Ano", "corte": "Corte (%)", "vence_brasil": "Vence no Brasil", "vence_regional": "Vence na região", "vence": "Vence nos dois"}))
display(SA.tabela(REGRA.drop(columns="util").assign(ano=REGRA["ano"].astype(str)), "Regra fixada de antemão: vitórias do modelo por ano (mínimo de 4 de 6)", numero="7", indice=False,
                  inteiros=["cortes", "vitorias_modelo", "minimo"], rotulos={"ano": "Ano", "cortes": "Cortes", "vitorias_modelo": "Vitórias do modelo", "minimo": "Mínimo", "veredito": "Veredito"}))


Ano,Corte (%),Vence no Brasil,Vence na região,Vence nos dois
2022,5,sim,sim,sim
2022,10,sim,sim,sim
2022,25,não,sim,não
2022,50,não,sim,não
2022,67,não,sim,não
2022,90,sim,sim,sim
2026,5,sim,sim,sim
2026,10,sim,sim,sim
2026,25,sim,sim,sim
2026,50,sim,sim,sim


Ano,Cortes,Vitórias do modelo,Mínimo,Veredito
2022,6,3,4,modelo não útil
2026,6,6,4,modelo útil


In [7]:
# Sensibilidade a n_u: mesma regra, com cada definição da taxa. A adotada vem dos ajustes do laço principal; só as outras são refeitas
ADOTADA = (TAB_ERROS.merge(NU[["ano", "corte", "razao"]], on=["ano", "corte"]).rename(columns={"razao": "n_u_sobre_restante_real"})
           .assign(definicao=A.DEFINICAO_TAXA)[["definicao", "ano", "corte", "metodo", "erro_brasil_pp", "mae_regional_pp", "n_u_sobre_restante_real"]])
LS, FALHAS_S = [], []
for definicao in A.DEFINICOES_TAXA:
    if definicao == A.DEFINICAO_TAXA:
        LS.extend(ADOTADA.to_dict("records"))
        continue
    for ano in ANOS:
        s = secoes[secoes.ano == ano].reset_index(drop=True)
        for pct in A.CORTES:
            rec = A.recebidas_ate(s, pct)
            base = A.montar_base(s, dfm, ano, rec, definicao, secoes_1t=S1T)
            real = A.alvo(base)
            ests = {"bruta": A.estimativa_bruta(base), "swing": A.estimativa_swing(base), "swing_1t": AD.estimativa_swing_1t(base)}
            try:
                bm = base.reindex(cd_modelo)
                if (ano, pct) not in CAND_CORTE:
                    raise np.linalg.LinAlgError("desenho indisponível")
                cand, df2, mapa, corte = CAND_CORTE[(ano, pct)]
                treino, resto = A.sobrescritas_1t(bm, mapa) if PARTES_1T else (None, None)
                pm = A.ajustar_parcial(cand, df2, mapa, bm.y_c.to_numpy(), bm.n_c.to_numpy(), bm.n_u.to_numpy(), lat, lon, corte, treino, resto)
                ests["modelo"] = A.estimativa_modelo(base, pd.Series(pm.pi, index=bm.index))
            except (np.linalg.LinAlgError, PerfectSeparationError, ValueError, FloatingPointError) as e:
                FALHAS_S.append({"definicao": definicao, "ano": ano, "corte": pct, "erro": repr(e)[:120]})
            restante = float((base.n_final - base.n_c).sum())
            for met, est in ests.items():
                eb, mr = A.erros(est, real)
                LS.append({"definicao": definicao, "ano": ano, "corte": pct, "metodo": met, "erro_brasil_pp": eb, "mae_regional_pp": mr,
                           "n_u_sobre_restante_real": float(base.n_u.sum()) / restante})
SENS = pd.DataFrame(LS)
REGRA_S = pd.concat([A.aplicar_regra(g).assign(definicao=d) for d, g in SENS.groupby("definicao")], ignore_index=True)
REGRA_S["veredito"] = np.where(REGRA_S.util, "modelo útil", "modelo não útil")
SENS.to_csv(TAB / "apuracao_sensibilidade_nu.csv", index=False)
if FALHAS_S:
    display(SA.tabela(pd.DataFrame(FALHAS_S), "Ajustes da sensibilidade que falharam", numero="F2", indice=False))
nu_razao = NU.pivot(index="corte", columns="ano", values="razao").rename_axis("Corte (%)")
nu_razao.columns = [f"n_u / restante real {a}" for a in nu_razao.columns]
display(SA.tabela(nu_razao, "n_u estimado sobre o restante real, por corte (taxa de todos os válidos)", numero="8", casas=3))


,n_u / restante real 2022,n_u / restante real 2026
Corte (%),,
5,"0,997","0,961"
10,"0,997","0,960"
25,"0,997","0,958"
50,"0,996","0,956"
67,"0,996","0,955"
90,"0,995","0,952"


In [8]:
G.mostrar_e_salvar(G.erro_por_corte(TAB_ERROS, ANOS, "erro_brasil_pp", "erro absoluto do Brasil (p.p.)", "Erro do Brasil por corte e método (escala log)"), "apuracao_erro_brasil")
G.mostrar_e_salvar(G.erro_por_corte(TAB_ERROS, ANOS, "mae_regional_pp", "MAE regional (p.p.)", "MAE regional por corte e método (escala log)"), "apuracao_mae_regional")

**Leitura do gráfico.** Quanto mais baixa a linha, melhor o método; a escala é logarítmica, então uma distância vertical é uma razão de erros. O modelo é a linha mais baixa na maior parte dos cortes, mas a comparação decisiva é com o swing do 1º turno (tracejado laranja), que usa a mesma informação do 1º turno. **No MAE regional** o modelo fica abaixo das três linhas de base nos 12 cortes, de 1,1 vez (2022 aos 50%: 0,061 contra 0,068) a 8,5 vezes menor que a melhor linha de base; a bruta e o swing ficam sempre atrás do swing do 1º turno, e o modelo vai de 0,14 a 0,01 em 2022 e de 0,16 a 0,01 em 2026. **No erro do Brasil** a história é outra em 2022: o modelo (0,08 / 0,13 / 0,15 / 0,10 / 0,07 / 0,02 p.p.) fica acima do swing do 1º turno (0,39 / 0,24 / 0,13 / 0,08 / 0,06 / 0,02) aos 25, 50 e 67% (0,15 contra 0,13, 0,10 contra 0,08 e 0,07 contra 0,06) e empata aos 90% (0,0219 contra 0,0222). Em 2026 o modelo vence o swing do 1º turno nos 6 cortes, de 1,7 a 13 vezes (0,16 a 0,00 p.p. contra 0,27 a 0,04). Contra a bruta (4,57 a 0,38 em 2022; 0,22 a 2,37 em 2026) e o swing uniforme (0,13 a 1,62 em 2022; 0,14 a 1,19 em 2026) o modelo é pelo menos 4,6 vezes melhor nos 12 cortes, nos dois gráficos. O ponto mais apertado do Brasil é 2022 aos 50% (modelo 0,10 contra 0,08). Em 2022, portanto, a vantagem do modelo sobre o swing do 1º turno é regional, não nacional; em 2026 vale nas duas medidas.

In [9]:
for ano in ANOS:
    G.mostrar_e_salvar(G.estimativa_por_corte(IC, f"Estimativa do modelo com IC 95% por corte, {ano}", ano=ano), f"apuracao_estimativa_corte_{ano}")

**Leitura do gráfico.** Em 2022 o IC 95% do Brasil contém o resultado final (50,90%) nos 6 cortes; a largura é de 0,66 p.p. aos 5% (a maior), 0,53 aos 10% e 0,08 aos 90%. A estimativa fica entre 50,92% e 51,05%, sempre um pouco acima do final (no máximo 0,15 p.p.). Em 2026 também contém o final (51,78%) nos 6 cortes: a estimativa aos 5% é 51,62%, 0,16 p.p. abaixo, com IC de 0,66 p.p. (a maior largura é 0,75, aos 10%); de 10% em diante fica a no máximo 0,11 p.p. do final. Nas regiões o final está dentro do IC nos 60 de 60 casos (5 regiões × 6 cortes × 2 anos; tabela 5), com larguras de 0,01 a 1,61 p.p.; o Sudeste tem o IC mais largo em todos os cortes. Como o desenho sai só do apurado de cada corte, o IC não usa informação do resultado final. A largura média do Brasil é de 0,39 p.p. em 2022 e 0,38 em 2026 (no HEAD, sem o 1º turno do mesmo ano, 2,18 e 1,47): a variância do choque de UF caiu e com ela σ_UF (ver diagnóstico abaixo). O choque de UF é σ̂·T_k (Student, k = 24 a 26 UFs), o que alarga um pouco o intervalo em relação a tratar σ̂ como conhecido.

In [10]:
for ano in ANOS:
    curva = pd.read_csv(TAB / f"apuracao_curva_{ano}.csv")
    est = IC[(IC.ano == ano) & (IC.regiao == "Brasil")]
    G.mostrar_e_salvar(G.curva_apuracao(curva[curva.pct_recebido >= 1], est, float(est.real_pct.iloc[0]), f"Apuração bruta e estimativas do modelo, Brasil {ano}"), f"apuracao_curva_{ano}")

**Leitura do gráfico.** Em 2022 a contagem bruta parte de 46,3% aos 5% apurados e só se aproxima do resultado final (50,90%) no fim; a virada de Lula sobre Bolsonaro ocorre em 68,1% (50,0%). As estimativas do modelo ficam entre 50,9% e 51,1% desde o primeiro corte, todas um pouco acima do final. Em 2026 a bruta fica abaixo do final (51,78%), com 50,4% aos 56,6% apurados, e as estimativas do modelo ficam entre 51,6% e 51,8%. A diferença entre as duas é o que o modelo acrescenta: a ordem de chegada dos boletins não é aleatória.

In [11]:
import geopandas as gpd
s26 = secoes[secoes.ano == 2026]
# Corte de 10 %: mostra quais municípios já entraram na amostra de ajuste.
# Cor escura: município quase todo contado. Cor clara: ainda sem seções.
CORTE_MAPA = 10
rec = A.recebidas_ate(s26, CORTE_MAPA)
contado = s26[rec].groupby("cd_municipio_tse")["validos"].sum()
total_mun = s26.groupby("cd_municipio_tse")["validos"].sum()
pct_mun = (100 * contado / total_mun).reindex(cd_modelo).fillna(0.0)
geo_s = BD.gdf.copy(); geo_s["geometry"] = geo_s.geometry.simplify(0.12)
hov = (dfm["nome_municipio"].astype(str) + " (" + dfm["sigla_uf"].astype(str) + ")").tolist()
G.mapa(geo_s, pct_mun, f"% dos válidos contados por município, corte de {CORTE_MAPA}% (2026)", "apuracao_mapa_contado_2026", "% contado",
       escala=[[0, "#e8f0fb"], [1, "#184f95"]], zmin=0, zmax=100, hover=hov)

**Leitura do gráfico.** Aos 10% do total nacional, em 2026, 62,1% dos municípios ainda não tinham nenhuma seção recebida e só 572 (10,3%) tinham metade ou mais dos votos válidos contados (183 já completos). Pesada por votos, a parcela contada é de 31,2% no Norte e 20,9% no Centro-Oeste, contra 2,8% no Sudeste e 8,3% no Nordeste. O modelo ajustado nesse corte vê, portanto, poucos municípios e uma amostra enviesada para as regiões que apuram primeiro; por isso o corte de 5% e 10% é o mais incerto.

## Pressuposto declarado: as seções apuradas representam as não apuradas?

- O modelo supõe que o restante de um município tem o mesmo share que o apurado, dado o que as covariáveis dizem do município.
- Isso falha quando a ordem de chegada se correlaciona com o voto dentro do município. Exemplo: capitais grandes, onde zonas centrais e periferias chegam em tempos diferentes.
- A medida é a diferença entre o share apurado (ou o π̂ do modelo) e o share **real do restante**.
- As covariáveis do 1º turno do mesmo ano entram no π̂; a seleção das seções dentro do município continua fora do alcance de um modelo municipal e é medida a seguir e no diagnóstico (D4).
- Ela é calculada nos 20 maiores municípios (por válidos finais) e nos demais.
- A média é ponderada pelos votos restantes reais.
- Entram só municípios com apuração iniciada e restante > 0.

In [12]:
G.mostrar_e_salvar(G.representatividade_por_corte(REP, "Lacuna absoluta entre o share apurado (ou π̂) e o share real do restante"), "apuracao_representatividade")

**Leitura do gráfico.** Cada linha mede o quanto o share que a apuração mostra (cinza) ou o π̂ do modelo (azul) se afasta do share real dos votos ainda não contados, separando as 20 maiores cidades das demais; quanto mais perto de zero, mais representativo. O π̂ fica entre 0,30 e 0,68 p.p. nos dois grupos e nos 12 cortes, com as linhas das maiores e das demais quase juntas. O share apurado erra mais que o π̂ em todos os cortes, nos dois grupos. Nas 20 maiores o apurado vai de 0,73 a 4,38 p.p., com pico de 4,38 aos 10% de 2026, e o ganho do π̂ some quase todo a partir de 50% (0,73 e 0,75 p.p. do apurado contra 0,40 e 0,48). Nas demais o apurado vai de 2,8 a 6,9 p.p. (6,88 aos 5% de 2026; em 2022, de 3,2 a 5,8) e o π̂ fica em 0,44 a 0,64: é aí que a apuração parcial mais engana e onde o modelo mais corrige.

## O que governa a largura dos intervalos e a sensibilidade regional

- O IC mede quatro fontes de variância: o piso binomial dos votos restantes, β̂ (Paula 4.6.6), o choque de UF (σ̂·T_k, Student) e a sobredispersão θ. A de n_u entra na versão com n_u simulado.
- Antes do 1º turno do mesmo ano (HEAD) os IC eram largos (Brasil 1,82 p.p. em média sobre os 12 cortes; hoje 0,39) e o erro regional dependia de onde estavam as seções apuradas. Os diagnósticos abaixo mostram o que mudou.
- O restante é previsto fora do suporte do ajuste quando a alavancagem v₀ = x₀'(X'ŴX)⁻¹x₀ passa a máxima das linhas ajustadas (Paula 1.7.2 e 2.8.1): β̂ e UF então dominam.
- A ordem de apuração escolhe seções dentro do município; se a parte apurada já difere da restante no 1º turno, o modelo municipal não vê essa seleção. Mede-se a lacuna logit(share_c) − logit(share_u), em municípios e em UFs.
- Linha de base com o 1º turno: o swing do 1º para o 2º turno das partes apuradas aplicado ao 1º turno das restantes. É a linha de base `swing_1t` do veredito (doc 11).

In [13]:
# Diagnóstico por (ano, corte): D1 variância por componente, D2 composição, D3 extrapolação, D4 seleção dentro do município e swing do 1º turno.
ibge = pd.read_parquet(ROOT / "data/interim/ibge_municipios.parquet")
CAPITAIS = set(ibge.loc[ibge.id_municipio.astype("int64").isin(AD.CAPITAIS_IBGE), "id_municipio_tse"].astype("int64"))
assert len(CAPITAIS) == 27, len(CAPITAIS)
S1T_DIAG = S1T if S1T is not None else pd.read_parquet(ROOT / "data/interim/tse_bweb_1t.parquet")
REG_MOD = pd.DataFrame(LIN_REG).query("metodo == 'modelo'")
regiao_m = dfm["regiao"].astype(str).to_numpy()
D1, D2, D3, D4 = [], [], [], []
for ano in ANOS:
    s = secoes[secoes.ano == ano].reset_index(drop=True)
    for pct in A.CORTES:
        if (ano, pct) not in CAND_CORTE:
            continue
        rec = A.recebidas_ate(s, pct)
        base = A.montar_base(s, dfm, ano, rec, secoes_1t=S1T)
        base_1t = base if PARTES_1T else A.montar_base(s, dfm, ano, rec, secoes_1t=S1T_DIAG)
        info = {"ano": ano, "corte": pct}
        bm = base.reindex(cd_modelo)
        cand, df2, mapa, corte_km = CAND_CORTE[(ano, pct)]
        des = DESENHOS[(ano, pct)]
        treino, resto = A.sobrescritas_1t(bm, mapa) if PARTES_1T else (None, None)
        pm = A.ajustar_parcial(cand, df2, mapa, bm.y_c.to_numpy(), bm.n_c.to_numpy(), bm.n_u.to_numpy(), lat, lon, corte_km, treino, resto)
        pi = pd.Series(pm.pi, index=bm.index)
        pi_uf = A.share_uf_modelo(base, pi)
        fk = A.fatores_nu(base, B, SEMENTE)
        var = AD.decompor(AD.variancias_componentes(pm, bm, base, pi_uf, LIGACAO, des.sigma_uf, B, SEMENTE, fk, gl_uf=des.gl_sigma_uf))
        pic = pd.Series(A.completar_por_uf(pi.reindex(base.index), base["n_u"], base["uf"]))
        ic_nb = IC[(IC.ano == ano) & (IC.corte == pct)].set_index("regiao")
        for u in var.index:
            mf = AD.meia_largura_formula(base, pic, des.sigma_uf, None if u == "Brasil" else u, gl_uf=des.gl_sigma_uf)
            D1.append({**info, "unidade": u, "sigma_uf": des.sigma_uf, **var.loc[u].to_dict(),
                       "meia_uf_pp": float(1.96 * np.sqrt(max(var.loc[u, "inc_uf"], 0)) * 100),
                       "meia_formula_pp": mf[0], "meia_formula_com_f_pp": mf[1], "meia_ic_notebook_pp": float(ic_nb.loc[u, "largura_pp"] / 2)})
        fundir = ";".join("+".join(g) for g in des.termos.regiao_fundir)
        for _, x in AD.composicao_do_corte(base, CAPITAIS, fundir).iterrows():
            D2.append({**info, **x.to_dict()})
        n_u = np.nan_to_num(bm.n_u.to_numpy(float))
        cov, Xf, Xa = AD.informacao_do_ajuste(cand, df2, mapa, bm.y_c.to_numpy(), bm.n_c.to_numpy(), bm.n_u.to_numpy(), treino, resto)
        v_aj, v_all = AD.alavancagem(Xf, cov), AD.alavancagem(Xa, cov)
        ext = AD.extrapolacao_por_regiao(v_aj, v_all, n_u, regiao_m)
        brasil = {"regiao": "Brasil", "v_max_ajuste": float(v_aj.max()), "v0_max_resto": float(v_all[n_u > 0].max()),
                  "extrapolado_nu": float(n_u[v_all > v_aj.max()].sum() / n_u.sum())}
        er = REG_MOD[(REG_MOD.ano == ano) & (REG_MOD.corte == pct)].set_index("regiao").erro_pp
        for x in [*ext.to_dict("records"), brasil]:
            D3.append({**info, **x, "erro_modelo_pp": float(er.get(x["regiao"], np.nan))})
        # D4 e swing do 1º turno
        for tipo, t in (("municipal", AD.selecao_municipal(base_1t)), ("uf", AD.selecao_uf(s, S1T_DIAG, rec, ano, base_1t))):
            for u, x in t.iterrows():
                D4.append({**info, "unidade": u, "tipo": tipo, **x.to_dict()})
        real = A.alvo(base_1t)
        est = {"swing_1t": AD.estimativa_swing_1t(base_1t), "modelo": A.estimativa_modelo(base, pi)}
        for tipo, e in est.items():
            for u in real.index:
                D4.append({**info, "unidade": u, "tipo": tipo, "erro_pp": 100 * (e[u] - real[u])})
DIAG_VAR, DIAG_COMP, DIAG_EXT, DIAG_SEL = (pd.DataFrame(x) for x in (D1, D2, D3, D4))
for nome, t in (("variancia", DIAG_VAR), ("composicao", DIAG_COMP), ("extrapolacao", DIAG_EXT), ("selecao", DIAG_SEL)):
    t.to_csv(TAB / f"apuracao_diag_{nome}.csv", index=False)
# Painel 1: parte de cada componente na variância do Brasil; painel 2: lacuna do 1º turno (média municipal) contra o erro do modelo, por região
import plotly.graph_objects as go
from plotly.subplots import make_subplots
vb = DIAG_VAR[DIAG_VAR.unidade == "Brasil"]
rot = [f"{a} {c}%" for a, c in zip(vb.ano, vb.corte)]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Parte de cada componente na variância do Brasil", "Lacuna do 1º turno × erro do modelo, por região"), horizontal_spacing=0.12)
for c, nome in zip(("piso", *AD.COMPONENTES), ("Piso binomial", "β̂", "Choque de UF", "θ", "n_u")):
    fig.add_bar(x=rot, y=vb[f"parte_{c}"], name=nome, row=1, col=1)
fig.update_layout(barmode="stack")
gm = DIAG_SEL[(DIAG_SEL.tipo == "municipal") & (DIAG_SEL.unidade != "Brasil")][["ano", "corte", "unidade", "gap"]]
em = DIAG_SEL[(DIAG_SEL.tipo == "modelo") & (DIAG_SEL.unidade != "Brasil")][["ano", "corte", "unidade", "erro_pp"]]
gg = gm.merge(em, on=["ano", "corte", "unidade"])
for ano, cor in zip(ANOS, (G.COR_REFERENCIA, G.COR_PONTO)):
    g = gg[gg.ano == ano]
    fig.add_scatter(x=g.gap, y=g.erro_pp, mode="markers", name=str(ano), marker={"color": cor}, text=[f"{u}, {c}%" for u, c in zip(g.unidade, g.corte)], row=1, col=2)
fig.update_xaxes(title_text="Lacuna logit(apurado) − logit(restante), 1º turno", row=1, col=2)
fig.update_yaxes(title_text="Erro do modelo (p.p., com sinal)", row=1, col=2)
fig.update_yaxes(title_text="Parte da variância com tudo ligado", row=1, col=1)
G.mostrar_e_salvar(fig, "apuracao_diagnostico", altura=520)

**Leitura do gráfico.** À esquerda, a parte de cada componente na variância do Brasil (as partes não somam exatamente 1 porque os componentes não são aditivos). O choque de UF é o maior em 9 dos 12 cortes (62% a 73% em 2022; 35% a 52% em 2026) e n_u é o maior em 2026 aos 5%, 25% e 50% (53%, 60% e 48%). β̂ pesa de 3% a 12%, θ de 1% a 23% (mais nos cortes tardios de 2026) e o piso binomial menos de 0,5%. No HEAD o choque de UF explicava de 75% a 100% da variância em todos os cortes e n_u de 1% a 10%. Hoje σ_UF vale 0,011 a 0,027 (antes 0,052 a 0,137) e a variância de n_u quase não mudou: n_u virou o piso, e sua parte vai de 20% a 60%. Isso explica o intervalo mais estreito (meia-largura do Brasil de 0,04 a 0,37 p.p., antes 0,12 a 1,76) e uma sensibilidade regional menor: o termo do 1º turno do mesmo ano absorveu a heterogeneidade entre UFs. O n_u extrapolado (D3) é nulo em todos os cortes, como antes: a largura não vem de previsão fora do suporte do ajuste. À direita, cada ponto é uma região em um corte: a lacuna do 1º turno entre o apurado e o restante (−0,21 a 0,28) quase não se relaciona com o erro do modelo (−0,26 a 0,35 p.p.; r = 0,08 no conjunto). Só em 2022 há associação moderada (r = 0,47), com erros quase todos positivos.

In [14]:
# Uma linha por ano × corte (Brasil): maior componente da variância, peso das capitais no apurado, n_u extrapolado, lacuna do 1º turno e swing do 1º turno
t_var = DIAG_VAR[DIAG_VAR.unidade == "Brasil"].set_index(["ano", "corte"])
t_comp = DIAG_COMP[DIAG_COMP.regiao == "Brasil"].set_index(["ano", "corte"])
t_ext = DIAG_EXT[DIAG_EXT.regiao == "Brasil"].set_index(["ano", "corte"])
sel = DIAG_SEL[DIAG_SEL.unidade == "Brasil"]
sel_m, sel_u = (sel[sel.tipo == k].set_index(["ano", "corte"]) for k in ("municipal", "uf"))
erro_sw = DIAG_SEL[DIAG_SEL.tipo == "swing_1t"].assign(abs_pp=lambda d: d.erro_pp.abs())
sw_1t = pd.DataFrame({"sw_brasil": erro_sw[erro_sw.unidade == "Brasil"].set_index(["ano", "corte"]).abs_pp,
                      "sw_mae": erro_sw[erro_sw.unidade != "Brasil"].groupby(["ano", "corte"]).abs_pp.mean()})
mod = TAB_ERROS[TAB_ERROS.metodo == "modelo"].set_index(["ano", "corte"])
TAB_DIAG = pd.DataFrame({
    "Maior componente": t_var.maior_componente,
    "Parte do maior": [t_var.loc[i, f"parte_{c}"] for i, c in t_var.maior_componente.items()],
    "Capitais no apurado": t_comp.capitais_do_apurado,
    "n_u extrapolado": t_ext.extrapolado_nu,
    "Lacuna |1º turno| municipal": sel_m.gap_abs,
    "Lacuna |1º turno| UF": sel_u.gap_abs,
    "Erro Brasil swing 1º turno": sw_1t.sw_brasil,
    "Erro Brasil modelo": mod.erro_brasil_pp,
    "MAE regional swing 1º turno": sw_1t.sw_mae,
    "MAE regional modelo": mod.mae_regional_pp,
}).reset_index().rename(columns={"ano": "Ano", "corte": "Corte (%)"})
display(SA.tabela(TAB_DIAG.assign(Ano=TAB_DIAG["Ano"].astype(str)), "Diagnóstico por ano e corte: variância, composição do apurado, seleção dentro do município e swing do 1º turno", numero="9", casas=3, indice=False))

Ano,Corte (%),Maior componente,Parte do maior,Capitais no apurado,n_u extrapolado,Lacuna |1º turno| municipal,Lacuna |1º turno| UF,Erro Brasil swing 1º turno,Erro Brasil modelo,MAE regional swing 1º turno,MAE regional modelo
2022,"5,000",uf,"0,650","0,571","0,000","0,209","0,181","0,393","0,076","0,387","0,144"
2022,"10,000",uf,"0,619","0,442","0,000","0,176","0,129","0,237","0,131","0,236","0,099"
2022,"25,000",uf,"0,725","0,311","0,000","0,155","0,104","0,125","0,149","0,133","0,096"
2022,"50,000",uf,"0,719","0,260","0,000","0,135","0,115","0,080","0,104","0,068","0,061"
2022,"67,000",uf,"0,627","0,251","0,000","0,138","0,118","0,063","0,072","0,047","0,040"
2022,"90,000",uf,"0,720","0,242","0,000","0,155","0,136","0,022","0,022","0,015","0,011"
2026,"5,000",nu,"0,526","0,520","0,000","0,275","0,208","0,273","0,160","0,525","0,165"
2026,"10,000",uf,"0,517","0,460","0,000","0,260","0,226","0,226","0,113","0,496","0,128"
2026,"25,000",nu,"0,598","0,327","0,000","0,219","0,179","0,204","0,084","0,419","0,094"
2026,"50,000",nu,"0,483","0,236","0,000","0,119","0,135","0,141","0,037","0,268","0,038"


**Leitura da tabela.** (Brasil, por ano e corte.) O maior componente da variância é o choque de UF em 9 dos 12 cortes (parte de 0,35 a 0,73) e n_u em 2026 aos 5%, 25% e 50%. Os capitais pesam 0,57 (2022) e 0,52 (2026) do apurado aos 5% e 0,21 a 0,24 aos 90%. O n_u extrapolado é 0,000 em todas as linhas. A lacuna do 1º turno entre o apurado e o restante é de 0,09 a 0,28 logit nos municípios e de 0,10 a 0,23 nas UFs, e cai com o corte; são os mesmos valores do HEAD, porque dependem só dos dados do 1º turno e não do modelo (por região, a lacuna de UF é negativa em 46 de 60 casos). O erro do modelo caiu de 0,03–1,36 p.p. (MAE regional no HEAD) para 0,01–0,16 com a lacuna igual: a seleção dentro do município não é o que limitava o modelo, e sim a falta do 1º turno como covariável. Isso é uma inferência: o diagnóstico não isola a causa. Swing uniforme do 1º turno (linha de base do veredito): no erro do Brasil o modelo tem erro menor em 9 dos 12 cortes (em 2022 perde aos 25, 50 e 67% por 0,01 a 0,02 p.p.; aos 90% ganha por 0,0003, 0,0219 contra 0,0222); no MAE regional o modelo é menor nos 12 cortes (0,01 a 0,16 contra 0,02 a 0,52 do swing). O swing do 1º turno erra mais por região que no Brasil, e com sinais opostos: em 2026 aos 5% o erro é de −0,86 p.p. no Sudeste e +0,66 no Nordeste.

## Leitura

**Veredito: 2022 modelo não útil (3 de 6 cortes); 2026 modelo útil (6 de 6).** A regra: o modelo vence um corte só se tiver erro menor que **todas** as linhas de base (contagem bruta, swing uniforme e swing uniforme do 1º turno, `swing_1t`) **e** nas duas medidas (erro do Brasil e MAE regional); é útil com pelo menos 4 de 6 cortes, em cada ano. O `swing_1t` entrou como linha de base depois do primeiro resultado (registrado no doc 11): antes dele, o veredito era útil nos dois anos, com 6 de 6.

- **2022: 3 de 6, modelo não útil.** O modelo vence os cortes de 5, 10 e 90%. Perde no erro do Brasil aos 25, 50 e 67% para o swing do 1º turno (0,15 contra 0,13; 0,10 contra 0,08; 0,07 contra 0,06 p.p.) e vence o MAE regional nos 6 cortes. Erro do Brasil do modelo: 0,08 / 0,13 / 0,15 / 0,10 / 0,07 / 0,02 p.p. nos cortes 5 a 90, contra 4,57 a 0,38 da bruta, 0,13 a 1,62 do swing e 0,39 a 0,02 do swing do 1º turno. MAE regional: 0,14 / 0,10 / 0,10 / 0,06 / 0,04 / 0,01. Portanto, em 2022 a vantagem do modelo sobre o swing do 1º turno é regional, não nacional.
- **2026: 6 de 6, modelo útil.** Erro do Brasil: 0,16 / 0,11 / 0,08 / 0,04 / 0,02 / 0,00 p.p., contra 0,22 a 2,37 da bruta, 0,14 a 1,19 do swing e 0,27 a 0,04 do swing do 1º turno. MAE regional: 0,16 / 0,13 / 0,09 / 0,04 / 0,02 / 0,01 (swing do 1º turno: 0,52 a 0,04).
- Contra a bruta e o swing uniforme o modelo vence os 12 cortes, no Brasil e nas regiões. A derrota de 2022 vem só do swing do 1º turno, que usa a mesma informação do 1º turno do mesmo ano (`lag1t`). Essa leitura é uma dedução dos números: o diagnóstico não isola a causa.
- **1º turno do mesmo ano.** `lag1t` entra no desenho de todos os cortes. Sobre os 12 cortes, o erro médio do Brasil caiu de 0,42 para 0,08 p.p., o MAE regional médio de 0,54 para 0,08, a largura média do IC do Brasil de 1,82 para 0,39 e a das regiões de 2,67 para 0,55; σ_UF caiu de 0,052–0,137 para 0,011–0,027. O 1º turno é conhecido antes do 2º (regra de informação, doc 09).
- **Vazamento corrigido.** O desenho do modelo (covariáveis, regiões fundidas, dispersão, SLX, corte de Conley e σ_UF) era o do `02-MLG`, escolhido com o resultado final de 2026; agora é refeito em cada corte só com o apurado. Esse refazimento tinha piorado os cortes iniciais de 2022 (5%: 0,32 para 0,72 p.p.; 10%: 0,72 para 1,19) e alargado os IC; com o 1º turno os erros ficaram em 0,08 e 0,13. O teste de 2022 sempre foi honesto (o desenho nunca viu 2022); o vazamento favorecia 2026. O corte de Conley não cai mais no piso de 1 km: os 12 desenhos ficam entre 75 e 150 km.
- **Cobertura do IC 95% por unidade e ano** (cortes cobertos de 6, tabela 5, arquivo `apuracao_cobertura_unidade.csv`): as 12 unidades (Brasil e 5 regiões, 2 anos) ficam dentro em 6 de 6 (regiões 60 de 60; antes, 56). Simular a incerteza de n_u não mudou nenhuma contagem. A largura mudou em menos de 0,15 p.p. Os cortes são aninhados no tempo e as unidades são correlacionadas. Por isso a tabela é descritiva, não um teste de calibração. O choque de UF com T de Student (k = 24 a 26) alargou um pouco os IC e não mudou nenhuma contagem.
- **Taxa de n_u:** a leitura literal (só os dois candidatos do 1º turno) erra 6,6% (2022) e 9,5% (2026) por município. A de todos os válidos erra 2,1% e 4,1% e foi a adotada. A escolha usou a comparação com o n final, e isso é declarado. Mesmo assim, n_u subestima o restante em 0,3% (2022) a 4,8% (2026). **Sensibilidade:** com a taxa literal o veredito de 2022 é o mesmo (3 de 6, não útil) e o de 2026 cai para 5 de 6 (perde o Brasil aos 10%), ainda útil; a conclusão não depende da taxa.
- **Representatividade nas capitais:** os 20 maiores municípios concentram 24% a 46% do restante real. Nesse grupo, o share apurado erra o share do restante em 0,7 a 4,4 p.p. (2026 a 10%: apurado 3,6 p.p. abaixo do restante). Nos demais, o erro é de 2,8 a 6,9 p.p. O π̂ reduz a lacuna nos dois grupos nos 12 cortes (0,30 a 0,68 p.p.). A hipótese de que as seções apuradas representam as restantes dentro do município continua sem teste direto; o diagnóstico D4 mede a lacuna do 1º turno, e o modelo a absorve melhor que antes.


## Corte precoce (2 %): tabela própria, fora do veredito

O corte de 2 % responde a outra pergunta: o que já se diria com quase nada apurado. Ele não entra no veredito, na regra de aceitação (mínimo de 4 vitórias), na Tabela 5 nem nos agregados dos 12 cortes, que seguem só os cortes de `A.CORTES` (5, 10, 25, 50, 67 e 90 %). Motivo: a regra e o mínimo foram fixados antes do corte de 2 %, e incluí-lo mudaria o que o veredito mede.

- O estimador do IC é o mesmo dos demais cortes; a regra de calibração pré-registrada não disparou, então nada foi ajustado para o corte de 2 %.
- O desenho (termos, dispersão, SLX, corte de Conley e σ_UF) é refeito só com os municípios já apurados, como nos outros cortes.
- **Não estimável** significa que o desenho ou o ajuste do corte falhou (por exemplo, matriz singular ou região sem município apurado). A linha traz o motivo e o resultado final do Brasil; nenhuma estimativa é inventada nem entra em contagem de vitórias.
- A comparação é descritiva: modelo, contagem bruta e swing do 1º turno contra o resultado final, mais o maior componente da variância do IC (D1).


In [15]:
# Corte precoce: desenho (cache em results/cv), ajuste, IC, linhas de base e D1. Não toca nas listas do veredito (LIN, LIN_IC, ...).
PASTA_CV = ROOT / "results/cv"
LIN_PRECOCE = []
DADOS_P, BASES_P = {}, {}
for ano in ANOS:
    s = secoes[secoes.ano == ano].reset_index(drop=True)
    for pct in A.CORTES_PRECOCES:
        rec = A.recebidas_ate(s, pct)
        base = A.montar_base(s, dfm, ano, rec, secoes_1t=S1T)
        bm = base.reindex(cd_modelo)
        treino = A.sobrescritas_1t(bm, MAPAS[ano])[0]
        DADOS_P[(ano, pct)] = DP.dados_do_corte(dfm, MAPAS[ano], W, np.nan_to_num(bm.y_c.to_numpy(float)), np.nan_to_num(bm.n_c.to_numpy(float)), treino)
        BASES_P[(ano, pct)] = (base, bm, A.pct_efetivo(s, rec))
FALHAS_P = {}
DESENHOS_P = DP.desenhos_por_corte(DADOS_P, pasta=PASTA_CV, falhas=FALHAS_P)
for (ano, pct), (base, bm, pe) in BASES_P.items():
    real = A.alvo(base)
    ests = ic = d1 = ajuste = None
    motivo = ""
    try:
        des = DESENHOS_P.get((ano, pct))
        if des is None:
            raise np.linalg.LinAlgError(f"desenho indisponível: {FALHAS_P.get((ano, pct), '')}")
        assert des.gl_sigma_uf is not None   # cache sem a contagem de UFs do σ: apagar e refazer o desenho
        df2, mapa = S.adicionar_slx(dfm, MAPAS[ano], list(des.termos.slx), W)
        cand = Candidato("parcial", "betabin", LIGACAO, des.termos)
        treino, resto = A.sobrescritas_1t(bm, mapa)
        pm = A.ajustar_parcial(cand, df2, mapa, bm.y_c.to_numpy(), bm.n_c.to_numpy(), bm.n_u.to_numpy(), lat, lon, des.corte_km, treino, resto)
        pi = pd.Series(pm.pi, index=bm.index)
        pi_uf = A.share_uf_modelo(base, pi)
        ests = {"bruta": A.estimativa_bruta(base), "swing_1t": AD.estimativa_swing_1t(base), "modelo": A.estimativa_modelo(base, pi)}
        ic = A.intervalos(pm, bm, base, pi_uf, LIGACAO, des.sigma_uf, B, SEMENTE, gl_uf=des.gl_sigma_uf)
        fk = A.fatores_nu(base, B, SEMENTE)
        d1 = AD.decompor(AD.variancias_componentes(pm, bm, base, pi_uf, LIGACAO, des.sigma_uf, B, SEMENTE, fk, gl_uf=des.gl_sigma_uf))
        ajuste = A.AjustePrecoce(ests=ests, ic=ic, d1=d1)
    except np.linalg.LinAlgError as e:   # singular no ajuste: a linha diz "não estimável"; outro erro é bug e interrompe
        ests = ic = d1 = ajuste = None
        motivo = repr(e)[:200]
    LIN_PRECOCE += A.linhas_corte_precoce(A.CortePrecoce(ano=ano, corte=pct, pct_efetivo=pe), real, ajuste, motivo=motivo)


/home/ribamarneto/Documentos/Previsao Eleicoes/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1189: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = fit_method(
/home/ribamarneto/Documentos/Previsao Eleicoes/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1189: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = fit_method(
/home/ribamarneto/Documentos/Previsao Eleicoes/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1189: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = fit_method(
/home/ribamarneto/Documentos/Previsao Eleicoes/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1189: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = fit_method(


In [16]:
PRECOCE = pd.DataFrame(LIN_PRECOCE, columns=list(A.COLUNAS_PRECOCE))
PRECOCE.to_csv(TAB / "apuracao_precoce.csv", index=False)
display(SA.tabela(PRECOCE.assign(ano=PRECOCE["ano"].astype(str)), "Corte precoce (2 %): estimativa do modelo, IC 95%, linhas de base e resultado final, por região e ano", numero="10", casas=2,
                  indice=False, rotulos={"ano": "Ano", "corte": "Corte (%)", "pct_efetivo": "% efetivo", "regiao": "Região", "estado": "Estado", "motivo": "Motivo",
                                         "estimativa_pct": "Modelo (%)", "q025_pct": "IC 2,5% (%)", "q975_pct": "IC 97,5% (%)", "largura_pp": "Largura (p.p.)",
                                         "real_pct": "Final (%)", "bruta_pct": "Bruta (%)", "swing_1t_pct": "Swing 1º turno (%)", "dentro_ic95": "Final no IC",
                                         "maior_componente": "Maior componente", "parte_maior": "Parte do maior"}))


Ano,Corte (%),% efetivo,Região,Estado,Motivo,Modelo (%),"IC 2,5% (%)","IC 97,5% (%)",Largura (p.p.),Final (%),Bruta (%),Swing 1º turno (%),Final no IC,Maior componente,Parte do maior
2022,"2,00","2,01",Centro-Oeste,estimado,,"39,89","39,27","40,59","1,32","39,79","40,32","40,22",1.00,uf,"0,83"
2022,"2,00","2,01",Nordeste,estimado,,"69,40","68,91","69,89","0,97","69,34","66,12","69,94",1.00,uf,"0,76"
2022,"2,00","2,01",Norte,estimado,,"49,39","48,74","50,12","1,38","48,97","47,10","49,63",1.00,uf,"0,56"
2022,"2,00","2,01",Sudeste,estimado,,"45,40","44,59","46,24","1,65","45,74","48,75","46,18",1.00,uf,"0,92"
2022,"2,00","2,01",Sul,estimado,,"38,47","37,67","39,19","1,52","38,16","35,70","39,12",1.00,uf,"0,70"
2022,"2,00","2,01",Brasil,estimado,,"50,91","50,50","51,33","0,82","50,90","42,99","51,53",1.00,uf,"0,65"
2026,"2,00","2,00",Centro-Oeste,estimado,,"39,78","39,27","40,27","1,01","39,99","44,68","39,44",1.00,uf,"0,79"
2026,"2,00","2,00",Nordeste,estimado,,"69,48","69,13","69,85","0,72","69,37","70,73","70,02",1.00,uf,"0,82"
2026,"2,00","2,00",Norte,estimado,,"49,96","49,45","50,45","1,00","50,01","51,93","50,19",1.00,uf,"0,48"
2026,"2,00","2,00",Sudeste,estimado,,"46,55","45,88","47,26","1,38","47,24","49,59","46,36",1.00,uf,"0,63"
